# Paper 3 reproducibility notebook

Public, sanitized notebook for the aggregate methodological release `v1.0-paper3`.

This notebook reproduces metrics that can be calculated from public aggregate tables and clearly marks values that are documented only as restricted-workflow aggregates. It contains no personal records, addresses, coordinates, archival images, row-level discrepancies, credentials, or local author paths.

In [ ]:
from csv import DictReader
from pathlib import Path

ROOT = Path.cwd()
DATA = ROOT / 'paper3_metodos' / 'data'
if not DATA.exists():
    DATA = Path('..') / 'data'

def read_csv(name):
    with (DATA / name).open(newline='', encoding='utf-8') as fh:
        return list(DictReader(fh))

def pct(num, den):
    return round((num / den) * 100, 2)

summary = {row['metric']: row for row in read_csv('validation_summary.csv')}
fields = read_csv('concordance_by_field.csv')
years = read_csv('concordance_by_year.csv')
sensitivity = read_csv('sensitivity_summary.csv')
reviewers = read_csv('reviewer_agreement_summary.csv')
idem = read_csv('idem_resolution_summary.csv')
bootstrap = read_csv('bootstrap_summary.csv')

print(f'Data directory: {DATA.resolve()}')

## Corpus and independent validation sample

In [ ]:
corpus_total = int(summary['corpus_total']['value'])
sample_records = int(summary['independent_sample']['value'])
conceptual_comparisons = int(summary['conceptual_comparisons']['value'])

assert corpus_total == 1438
assert sample_records == 180
assert conceptual_comparisons == 1980

print({'corpus_total': corpus_total, 'sample_records': sample_records, 'conceptual_comparisons': conceptual_comparisons})

## Semantic/resolved agreement from public aggregate tables

In [ ]:
field_comparisons = sum(int(row['comparisons']) for row in fields)
field_matches = sum(int(row['matches']) for row in fields)
field_disagreements = sum(int(row['disagreements']) for row in fields)
agreement = pct(field_matches, field_comparisons)

assert field_comparisons == conceptual_comparisons
assert field_matches == 1867
assert field_disagreements == 113
assert agreement == 94.29

print({'matches': field_matches, 'comparisons': field_comparisons, 'semantic_resolved_agreement_percent': agreement})

## Annual composition-weighted agreement

In [ ]:
year_comparisons = sum(int(row['comparisons']) for row in years)
year_matches = sum(int(row['matches']) for row in years)
year_corpus = sum(int(row['corpus_records']) for row in years)

weighted = sum(
    (int(row['matches']) / int(row['comparisons'])) * (int(row['corpus_records']) / year_corpus)
    for row in years
)
weighted_percent = round(weighted * 100, 2)

assert year_corpus == corpus_total
assert year_comparisons == conceptual_comparisons
assert year_matches == 1867
assert weighted_percent == 94.41

print({'annual_composition_weighted_agreement_percent': weighted_percent})

## Sensitivity excluding empty-empty comparisons

In [ ]:
sens = {row['scenario']: row for row in sensitivity}
non_empty = sens['excluding_empty_empty']
non_empty_comparisons = int(non_empty['comparisons'])
non_empty_matches = int(non_empty['matches'])
excluded = int(non_empty['excluded_comparisons'])
non_empty_agreement = pct(non_empty_matches, non_empty_comparisons)

assert excluded == 295
assert non_empty_comparisons == 1685
assert non_empty_matches == 1572
assert non_empty_agreement == 93.29

print({'excluded_empty_empty': excluded, 'sensitivity_percent': non_empty_agreement})

## Double review and contextual repetition/IDEM checks

In [ ]:
review = reviewers[0]
review_records = int(review['records'])
review_comparisons = int(review['comparisons'])
review_matches = int(review['exact_matches'])
review_agreement = pct(review_matches, review_comparisons)

idem_row = idem[0]
idem_marks = int(idem_row['marks_evaluated'])
idem_correct = int(idem_row['resolved_correctly'])
idem_success = pct(idem_correct, idem_marks)

assert review_records == 60
assert review_comparisons == 1080
assert review_agreement == 100.00
assert idem_marks == 398
assert idem_correct == 397
assert idem_success == 99.75

print({'double_review_agreement_percent': review_agreement, 'idem_resolution_success_percent': idem_success})

## Bootstrap interval

The bootstrap interval is released as an aggregate from the restricted validation workflow. The row-level resampling inputs are not public because they belong to the restricted validation materials.

In [ ]:
boot = bootstrap[0]
assert float(boot['estimate_percent']) == 94.29
assert float(boot['ci95_lower_percent']) == 92.70
assert float(boot['ci95_upper_percent']) == 95.98

print({'bootstrap_ci95_percent': (boot['ci95_lower_percent'], boot['ci95_upper_percent'])})

## Interpretation note

These metrics are not CER, WER, or HTR engine accuracy. The public release evaluates an aggregate structured pipeline: transcription, contextual resolution, normalization, QA, and human validation. Correctly expanded `IDEM` values are valid under the original transcription contract.